In [0]:
# bootstrap_servers="pkc-75m1o.europe-west3.gcp.confluent.cloud:9092"
# api_keys="5NZ5TSOWHTM3TWBG"
# api_secret="cfltNAApWdJ9Z1FFkmQHnuMQzF2IBQzxeuQkgpBdHxJpSbue07FjjZByINl74WaA"
# topic="credit_card_transactions"

In [0]:
import json
kafka_connection_json=dbutils.secrets.get(scope="fineguard-scope",key="kafka_connection_details")

kafka_config=json.loads(kafka_connection_json)
bootstrap_servers=kafka_config["bootstrap_servers"]
api_keys=kafka_config["api_keys"]
api_secret=kafka_config["api_secret"]
topic=kafka_config["topic"]

In [0]:
jaas_config=f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="{api_keys}" password="{api_secret}";'

In [0]:
sample_batch = (spark.read
    .format("kafka")
    .option("kafka.bootstrap.servers", bootstrap_servers)
    .option("subscribe", topic)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config", jaas_config)
    .option("startingOffsets", "earliest")
    .load()
)

In [0]:
sample_batch.count()

In [0]:
display(sample_batch)

In [0]:
from pyspark.sql.functions import col

parsed_batch= sample_batch.select(col("key").cast("string"),col("value").cast("string"),col("topic"),col("partition"),col("offset"),col("timestamp"),col("timestampType"))

In [0]:
display(parsed_batch)

In [0]:
parsed_batch.write.saveAsTable("fineguard.bronze.transactions_batch_test")

In [0]:
streaming_df = (spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", bootstrap_servers)
    .option("subscribe", topic)
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config", jaas_config)
    .option("startingOffsets", "earliest")
    .load()
)

In [0]:
from pyspark.sql.functions import col

parsed_streaming_df= streaming_df.select(col("key").cast("string"),col("value").cast("string"),col("topic"),col("partition"),col("offset"),col("timestamp"),col("timestampType"))

In [0]:
streaming_query=parsed_streaming_df.writeStream.format("delta").outputMode("Append").option("checkPointLocation","/Volumes/fineguard/source/transactions/checkpoint/").trigger(availableNow=True).toTable("fineguard.bronze.transactions_streaming_test")